## Setup

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from PIL import Image
from imageio import *
import torch
from skimage.transform import resize
from mpl_toolkits.axes_grid1.axes_rgb import make_rgb_axes, RGBAxes
from torchvision.models import *
from torchvision.datasets import MNIST,KMNIST,FashionMNIST
from skimage.util import montage
import skimage.io as skio
from scipy import signal
from torchvision import transforms
import requests
from torchsummary import summary

In [ ]:
def plot(x):
    if type(x) == torch.Tensor :
        x = x.cpu().detach().numpy()

    fig, ax = plt.subplots()
    im = ax.imshow(x, cmap = 'gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
urllib.request.urlretrieve('https://raw.githubusercontent.com/imageio/imageio-binaries/master/images/imageio_banner.png',"image1.png")

im = Image.open("image1.png")

In [ ]:
im

In [ ]:
im = imread('https://raw.githubusercontent.com/imageio/imageio-binaries/master/images/imageio_banner.png')

In [ ]:
plot(im)

### Resize Image

In [ ]:
im.shape

In [ ]:
height = 128
width = 128

im_new = resize(im,(height,width))

In [ ]:
plot(im)

### Show RGB Channels

In [ ]:
im = imread('https://raw.githubusercontent.com/imageio/imageio-binaries/master/images/imageio_banner.png')

In [ ]:
im.shape

In [ ]:
plot(im)

In [ ]:
fig = plt.figure()
ax = RGBAxes(fig, [0.1, 0.1, 0.8, 0.8])
ax.imshow_rgb(im[:,:150,0],im[:,:150,1],im[:,:150,2]);

In [ ]:
im.shape

In [ ]:
red   = im[:,:,0]
green = im[:,:,0]
blue  = im[:,:,0]

blank = np.zeros_like(red)

In [ ]:
blank.shape

In [ ]:
red.shape

In [ ]:
#blank = np.zeros((red.shape[0],red.shape[1]))

In [ ]:
blank.shape

In [ ]:
plot(np.stack([red,blank,blank],-1))
plot(np.stack([blank,blue,blank],-1))
plot(np.stack([blank,blank,green],-1))

## Alexnet

In [ ]:
def scale(x):
    return (x - np.min(x))/np.max(x)
def scale1(x):
    return (x-np.min(x))/(np.max(x)-np.min(x))

In [ ]:
def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x,cmap='gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
image = skio.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")
image = image[:,0:700,:]
plot(image)

In [ ]:
coin = image[185:200,224:239,:]

In [ ]:
plot(coin)

In [ ]:
image = image[60:,0:700,:]
plot(image)

In [ ]:
image=np.mean(image,axis=2)
coin=np.mean(coin,axis=2)
image=scale1(image)
coin=scale1(coin)

In [ ]:
plot(image)


In [ ]:
plot(coin)

In [ ]:
coin.shape

In [ ]:
image = image - np.mean(image)
coin = coin - np.mean(coin)

In [ ]:
z=signal.convolve2d(image, np.rot90(coin, 2))

In [ ]:
plot(z)

In [ ]:
plot(z==np.max(z))

In [ ]:
[y,x] = np.where(z == np.amax(z))

In [ ]:
plt.plot(x,-y,'r.',markersize=30)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(image, cmap = 'gray')
ax.axis('off')
fig.set_size_inches(20, 10)
ax.scatter(x-6, y-6, c='#f97306', s=80) #https://xkcd.com/color/rgb/
plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Select GPU if available

In [ ]:
net = alexnet(pretrained=True).to(device)

In [ ]:
normalize = transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   normalize
])

In [ ]:
im = skio.imread('https://www.medicalnewstoday.com/content/images/articles/322/322868/golden-retriever-puppy.jpg')

In [ ]:
plot(im)

In [ ]:
image = Image.fromarray(im) #convert to pil

In [ ]:
img_tensor = preprocess(image)

In [ ]:
img_tensor = img_tensor.unsqueeze_(0)

In [ ]:
img_tensor.shape

In [ ]:
img_variable = torch.tensor(img_tensor).to(device)

In [ ]:
img_variable.shape

In [ ]:
%%timeit
out = net(img_variable)

In [ ]:
out = net(img_variable)

In [ ]:
out.shape

In [ ]:
out

In [ ]:
label_index = out.cpu().data.numpy().argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])

In [ ]:
labels_url = 'https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json'
labels = {int(key):value for (key, value) in requests.get(labels_url).json().items()}

In [ ]:
print(labels[label_index])

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

In [ ]:
net

In [ ]:
summary(net, (3, 224, 224))

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
plot(out[0,10,:,:])

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
im = skio.imread('http://bocasurfcam.com/most_recent_image.php')

In [ ]:
plot(im)

In [ ]:
def load_im(im):
    image = Image.fromarray(im) #convert to pil
    img_tensor = preprocess(image)
    img_tensor = img_tensor.unsqueeze_(0)
    img_variable = torch.tensor(img_tensor).to(device)
    return img_variable

In [ ]:
out = net(load_im(im))

In [ ]:
def inference(im):
    out = net(load_im(im))
    label_index = out.cpu().data.numpy().argmax()
    top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])
    print(labels[label_index])
    print('____')
    for i in range(10):
        print(labels[top_list[i]])

In [ ]:
inference(im)

In [ ]:
net

In [ ]:
w1 = net.features[0].weight.data.cpu().numpy()

In [ ]:
w1.shape

In [ ]:
w1 = np.transpose(w1,(0,2,3,1))

In [ ]:
w1.shape

In [ ]:
Wp = (w1 - np.mean(w1))/np.std(w1)

In [ ]:
M = montage(Wp, multichannel=True)

In [ ]:
M.shape

In [ ]:
plot(Wp[0,:,:,:])

In [ ]:
plot(np.random.random((11,11,3)))

In [ ]:
plot(montage(Wp, multichannel=True))

In [ ]:
plot(montage(w1, multichannel=True))

In [ ]:
plot(scale1(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
img_variable.flatten().shape

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
out.shape

In [ ]:
plot(out[0,0,:,:])

In [ ]:
img_variable.shape

In [ ]:
3*224*224

In [ ]:
64*55*55

In [ ]:
out = np.squeeze(out)

In [ ]:
plot(montage(out))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
for i in range(12):
    plot(montage(np.squeeze(net.features[0:i](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.avgpool(net.features[0:13](img_variable)).cpu().detach().numpy())))

In [ ]:
net.features[0:13](img_variable).cpu().detach().numpy().shape

In [ ]:
net

In [ ]:
net.avgpool(net.features[0:13](img_variable)).shape

In [ ]:
net.avgpool(net.features[0:13](img_variable)).flatten().shape[0]

In [ ]:
plt.plot(np.arange(9216),net.avgpool(net.features[0:13](img_variable)).flatten().cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
9216 / (3*224*224) # percentage of numbers left, vs original pixels

In [ ]:
net.classifier[0](net.avgpool(net.features[0:13](img_variable)).flatten()).shape

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:2](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:3](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:5](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(1000),net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
out = net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy()

In [ ]:
def softmax(x):

    s = np.exp(x)

    s = s / np.sum(s)

    return s

In [ ]:
out = softmax(out)

In [ ]:
plt.plot(np.arange(1000),out)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
label_index = out.argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out))[0:15]

In [ ]:
top_list

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

## Alexnet Custom

Two possible ways to create your dataset: (1) Make one with your own with your camera/phone or (2) Build one with Image Search (Google/Bing)

The idea is to have (at least) two categories for which you will train an inference model. You will then present data from your reserved validation set to evaluate your models ability to distinguish the two categories. For example, pick any living object like ants and then collect photos with and without ants in the pictures. You will then put these pictures on your google drive and run a classification model just like with flowers 102.

The more photos you collect the better your model will be. You must have at least 50 photographs for each of your two categories. (This should not take a long time.)

Note you must put around 20% of your images into the valid folder.

Example data folders here: https://drive.google.com/drive/folders/1Je_gRF8RnhszAAbQhw2IzqM2_7byPVtL?usp=sharing

Example Ideas:

Ex1 Camera: Ant finder Close up of ground/floor with ants in photo vs Close up photo with no ants

Ex2 Camera: Palm tree with coconuts vs Palm tree with dates (Both are all over FAU)

Ex3 Web Search: Marine animal (bird) vs Marine animal covered in oil from oil spill

Ex4 Web Search: Burrowing owl adult vs Burrowing owl chick

In [ ]:
!pip install flashtorch
!pip install barbar

In [ ]:
!pip install git+https://github.com/williamedwardhahn/mpcr
from mpcr import *
from flashtorch.utils import apply_transforms
from flashtorch.saliency import Backprop
import itertools
from sklearn.metrics import confusion_matrix
from torchsummary import summary

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Create new folder on your google drive like below:

In [ ]:
data_dir = '/content/drive/My Drive/Data1/antsbeesdataset/'

In [ ]:
os.chdir(data_dir)

In [ ]:
pwd

In [ ]:
ls

In [ ]:
ls train

In [ ]:
ls valid

In [ ]:
dataset_labels =["ants", "bees"] #<--- Replace with your labels

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),  #Data Augmentation
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

In [ ]:
batch_size = 16
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),data_transforms[x]) for x in ['train', 'valid']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=2) for x in ['train', 'valid']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset_sizes

In [ ]:
def imshow(inp, title = " "):
    fig, ax = plt.subplots()
    inp = inp.numpy().transpose((1, 2, 0))
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    ax.imshow(inp)
    plt.title(title, loc='center')
    # fig.set_size_inches(5, 5)
    plt.show()

In [ ]:
inputs, classes = next(iter(dataloaders['train']))
inputs = inputs[:4]
classes = classes[:4]
out = torchvision.utils.make_grid(inputs)
imshow(out, title=[dataset_labels[x] for x in classes])

In [ ]:
def train_model(model, num_epochs=25):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    for epoch in range(num_epochs):

        print('Epoch: ',epoch+1,'/',num_epochs)

        ###Train
        model.train()
        running_corrects = 0
        for inputs, labels in dataloaders['train']:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            preds = torch.max(outputs, 1)[1]
            running_corrects += torch.sum(preds == labels.data)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        print("Train ", 'Acc: {:.2f}'.format(running_corrects.double()/dataset_sizes['train']))

        scheduler.step()

        ###Val
        model.eval()
        running_corrects = 0
        for inputs, labels in dataloaders['valid']:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = torch.max(outputs, 1)[1]
            running_corrects += torch.sum(preds == labels.data)

        print("Valid ", 'Acc: {:.2f}'.format(running_corrects.double()/dataset_sizes['valid']))
        print("#######################")
    return model

In [ ]:
model = models.alexnet(pretrained=True)

In [ ]:
batch_size = 16
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),data_transforms[x]) for x in ['train', 'valid']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=2) for x in ['train', 'valid']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model

In [ ]:
model.classifier[6] = nn.Linear(4096, 102)

In [ ]:
model

In [ ]:
model = train_model(model, num_epochs=3)

In [ ]:
model = train_model(model, num_epochs=30)

In [ ]:
def visualize_model(model, num_images=16):
    model.eval()
    index = 0
    for i, (inputs, labels) in enumerate(dataloaders['valid']):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        preds = torch.max(outputs, 1)[1]

        for j in range(inputs.size()[0]):
            index += 1
            title1 = 'predicted: ' + dataset_labels[preds[j]] + '     class: ' + dataset_labels[labels[j]]
            imshow(inputs.cpu().data[j],title1)

            if index == num_images:
                return

In [ ]:
visualize_model(model)

In [ ]:
summary(model, (3, 224, 224))

In [ ]:
image = io.imread('https://cdn.branchcms.com/7XlaZw139w-1201/images/ant-pest-image.v1.jpg')
plt.imshow(image);

In [ ]:
img = apply_transforms(image).clone().detach().requires_grad_(True).to(device)

In [ ]:
outputs = model(img)
preds = torch.max(outputs, 1)[1]

In [ ]:
print('predicted: ' + dataset_labels[preds])

In [ ]:
batch_size = 1024 #large batch size so we test all the data
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),data_transforms[x]) for x in ['train', 'valid']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=4) for x in ['train', 'valid']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
inputs,labels = next(iter(dataloaders['valid']))
inputs = inputs.to(device)
labels = labels.detach().cpu().numpy()

In [ ]:
outputs = model(inputs)
preds = torch.max(outputs, 1)[1].detach().cpu().numpy()

In [ ]:
cm = confusion_matrix(preds.reshape(-1),labels)

In [ ]:
print("Confusion Matrix : ")
cm

## Load Data Deck

Google Slides URL: be sure to share slides with "Anyone with Link"

In [ ]:
url = "https://docs.google.com/presentation/d/1lyE52a0yLCu9iUiKeFmdKayGJhu2Z6fyToXEMKeigoY/edit#slide=id.p"

In [ ]:
%%capture
!apt-get install poppler-utils
!pip install pdf2image

In [ ]:
import requests
from pdf2image import convert_from_path
import matplotlib.pyplot as plt
import numpy as np

def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x, cmap = 'gray')
    ax.axis('off')
    fig.set_size_inches(5, 5)
    plt.show()

def get_slides(url):
    url_head = "https://docs.google.com/presentation/d/"
    url_body = url.split('/')[5]
    page_id = url.split('.')[-1]
    url = url_head + url_body + "/export/pdf?id=" + url_body + "&pageid=" + page_id
    r = requests.get(url, allow_redirects=True)
    open('file.pdf', 'wb').write(r.content)
    images = convert_from_path('file.pdf', 500)
    return images

In [ ]:
images = get_slides(url)

for i, image in enumerate(images):
    plot(image)
    print(np.array(image).shape)
    # image.save('out_'+str(i)+'.jpg', 'JPEG')

    img = apply_transforms(image).clone().detach().requires_grad_(True).to(device)
    outputs = net(img)
    preds = torch.max(outputs, 1)[1]
    preds = preds.item()
    LABELS_URL = 'https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json'
    labels = {int(key):value for (key, value) in requests.get(LABELS_URL).json().items()}
    print(labels[preds])